# Create / reset YAML files
This notebook is used to create / reset YAML files for all datasets. Existing YAML files are deleted and new "template" ones are created based on the config schema in MassiPipe. Each YAML file is then modified, filling in the details that are specific to each dataset (date, grouping, area, number of raw files). These details (except number of files) are inferred from the name of the dataset directory, which follows the pattern `massimal_<grouping>_<area>_<datetime>-<optional_identifier(s)>_hsi`. 

In [1]:
import yaml
from pathlib import Path  
import shutil
from massipipe.pipeline import (
    PipelineProcessor, read_config, write_config, export_template_yaml
)
from rich import print

In [2]:
# Parameters
base_dir = '/home/mha114/data/seabee-minio'
# base_dir = '/home/mha114/data/seabee-minio/larvik/kongsbakkebukta/aerial/hsi/20230830/massimal_larvik_kongsbakkebukta_202308301328_hsi/'

yaml_file_name = "config.seabee.yaml"

yaml_creator_name = "Martin H. Skjelvareid"
yaml_organisation = "UiT"
yaml_project = "Massimal"

In [3]:
def find_hsi_datasets(base_dir:Path,dataset_subdir_search_str='0_raw'):
    """Find HSI dataset paths based on expected subdirectory in dataset"""
    subdirs = base_dir.rglob(dataset_subdir_search_str)
    return [subdir.parent for subdir in subdirs]

In [4]:
# "Dry-run", show found dataset dirs
dataset_dirs = find_hsi_datasets(Path(base_dir))
for dd in dataset_dirs:
    print(dd.name)

massimal_smola_skalmen_202306201736-nw_hsi

massimal_smola_skalmen_202306201520-nw_hsi

massimal_smola_skalmen_202306201815-se_hsi

massimal_smola_skalmen_202306201640-nw_hsi

massimal_smola_skalmen_202306201552-nw_hsi

...

massimal_vega_sola_202208231608-coast3_hsi

massimal_vega_sola_202208231139-sound2_hsi

massimal_vega_sola_202208231509-coast1_hsi

massimal_vega_sola_202208231211-sound3a_hsi

massimal_vega_sola_202208231111-sound1_hsi

In [5]:
# Update YAML files with date, area etc. specific to each dataset
for dataset_dir in dataset_dirs:
    # Get dataset information from dataset directory name
    try:
        _,grouping,area,datetime,_ = dataset_dir.name.split('_')
        datetime = datetime.split('-')[0] # Remove additional dataset identifier, e.g. "north"
        nfiles = sum([Path(raw_dir).is_dir() for raw_dir in (dataset_dir / '0_raw').glob('*')])
    except ValueError:
        print(f'Error while parsing dataset {dataset_dir}')
        
    # Create config path
    config_path = dataset_dir / yaml_file_name
    
    # Delete any existing YAML file
    if config_path.exists():
        config_path.unlink()

    # Create new template YAML file
    export_template_yaml(config_path)

    # Update with information from directory name
    config = read_config(config_path)
    config['grouping'] = grouping
    config['area'] = area
    config['datetime'] = datetime
    config['nfiles'] = nfiles
    config['organisation'] = yaml_organisation
    config['creator_name'] = yaml_creator_name
    config['project'] = yaml_project
    
    write_config(config,config_path)